# JOILang Cloud-Only Advisor Compression Analysis

**Cell 0. Title and purpose**

This notebook traces **cloudless**, **hybrid advisor**, and **cloud-only advisor compression** modes for the JOILang GA prompt-compression framework. It validates dataset-feedback summarization, exact cloud advisor prompt sections, proposal normalization/rejection/repair, before/after prompt-block changes, token/DETPass artifacts, and cloud-only isolation from `compression_fallback`.

The notebook is an analysis/smoke notebook, not a full paper run. Full commands are prepared later with `RUN_FULL_* = False` by default.

In [3]:
# Cell 1. Server/path configuration
from __future__ import annotations
import csv, json, os, shutil, subprocess, sys
from datetime import datetime
from pathlib import Path

SERVER_PRESET = 'A6000_SET_A'  # Options: 'A100_SET_B', 'A6000_SET_A'

PRESETS = {
    'A100_SET_B': {
        'repo_hints': [
            Path('/root/llm/JOILang-Server'),
            Path.cwd(),
            Path.cwd().parent,
        ],
        'local_model_base': os.environ.get(
            'JOILANG_MODEL_BASE',
            '/root/llm/local_models',
        ),
        'python': Path('/root/llm/je/bin/python'),
    },
    'A6000_SET_A': {
        'repo_hints': [
            Path('/home/mgjeong/Desktop/llm/JOILang-Server'),
            Path.cwd(),
            Path.cwd().parent,
        ],
        'local_model_base': os.environ.get(
            'JOILANG_MODEL_BASE',
            '/home/mgjeong/Desktop/llm/local_models',
        ),
        'python': Path('/home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10'),
    },
}


def autodetect_repo_root():
    rel = Path('gpt_mg') / 'version0_15_update20260413' / 'scripts' / 'run_ga_search.py'
    candidates = []

    for hint in PRESETS[SERVER_PRESET]['repo_hints']:
        p = Path(hint)
        candidates.extend([p, *p.parents])

    candidates.extend([Path.cwd(), *Path.cwd().parents])

    seen = set()
    for c in candidates:
        s = str(c)
        if s in seen:
            continue
        seen.add(s)

        if (c / rel).exists():
            return c

    raise FileNotFoundError('JOILang-Server repo root not found')


REPO_ROOT = autodetect_repo_root()
VERSION_ROOT = REPO_ROOT / 'gpt_mg' / 'version0_15_update20260413'
SCRIPTS_ROOT = VERSION_ROOT / 'scripts'
RESULTS_ROOT = VERSION_ROOT / 'results'
NOTEBOOKS_ROOT = VERSION_ROOT / 'notebooks'

RUN_ID = 'cloud_only_advisor_analysis_' + datetime.now().strftime('%Y%m%d_%H%M%S')
ANALYSIS_ROOT = RESULTS_ROOT / RUN_ID
ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)

# Important: do not resolve this symlink; keep the executable exactly as configured.
# Using sys.executable can accidentally select the Jupyter kernel Python, so use server preset Python.
PYTHON_EXE = str(PRESETS[SERVER_PRESET]['python'])

MODEL_KEY = 'qwen25_coder_14b'
ADVISOR_MODEL_KEY = 'gpt41_mini'
SMOKE_CATEGORIES = (3, 4)
SMOKE_LIMIT_PER_CATEGORY = 1

USE_MOCK_LLM = True
RUN_SMOKE_COMMANDS = True
RUN_FULL_CLOUD_ONLY = False
RUN_FULL_THREE_WAY = False

LOCAL_MODEL_BASE = PRESETS[SERVER_PRESET]['local_model_base']

# Runtime environment for local model loading.
os.environ['JOI_V15_LOCAL_MODEL_BASE_DIR'] = str(LOCAL_MODEL_BASE)
os.environ['JOI_V15_LOCAL_FILES_ONLY'] = 'true'
os.environ['TRANSFORMERS_OFFLINE'] = os.environ.get('TRANSFORMERS_OFFLINE', '1')
os.environ['HF_HUB_OFFLINE'] = os.environ.get('HF_HUB_OFFLINE', '1')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print(json.dumps({
    'server_preset': SERVER_PRESET,
    'repo_root': str(REPO_ROOT),
    'version_root': str(VERSION_ROOT),
    'python_executable_raw': str(PYTHON_EXE),
    'local_model_base': str(LOCAL_MODEL_BASE),
    'analysis_root': str(ANALYSIS_ROOT),
    'model_key': MODEL_KEY,
    'advisor_model_key': ADVISOR_MODEL_KEY,
}, indent=2, ensure_ascii=False))

print('\n[PATH CHECK]')
for name, path in {
    'REPO_ROOT': REPO_ROOT,
    'VERSION_ROOT': VERSION_ROOT,
    'SCRIPTS_ROOT': SCRIPTS_ROOT,
    'RUN_GA_SEARCH': SCRIPTS_ROOT / 'run_ga_search.py',
    'PYTHON_EXE': Path(PYTHON_EXE),
    'LOCAL_MODEL_BASE': Path(LOCAL_MODEL_BASE),
    'RESULTS_ROOT': RESULTS_ROOT,
    'ANALYSIS_ROOT': ANALYSIS_ROOT,
}.items():
    print(f'{name:18s}: {path} exists={Path(path).exists()}')

assert REPO_ROOT.exists(), REPO_ROOT
assert VERSION_ROOT.exists(), VERSION_ROOT
assert SCRIPTS_ROOT.exists(), SCRIPTS_ROOT
assert (SCRIPTS_ROOT / 'run_ga_search.py').exists(), SCRIPTS_ROOT / 'run_ga_search.py'
assert Path(PYTHON_EXE).exists(), PYTHON_EXE
assert Path(LOCAL_MODEL_BASE).exists(), LOCAL_MODEL_BASE

{
  "server_preset": "A6000_SET_A",
  "repo_root": "/home/mgjeong/Desktop/llm/JOILang-Server",
  "version_root": "/home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413",
  "python_executable_raw": "/home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10",
  "local_model_base": "/home/mgjeong/Desktop/llm/local_models",
  "analysis_root": "/home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/cloud_only_advisor_analysis_20260612_132600",
  "model_key": "qwen25_coder_14b",
  "advisor_model_key": "gpt41_mini"
}

[PATH CHECK]
REPO_ROOT         : /home/mgjeong/Desktop/llm/JOILang-Server exists=True
VERSION_ROOT      : /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413 exists=True
SCRIPTS_ROOT      : /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts exists=True
RUN_GA_SEARCH     : /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py exists=Tr

In [4]:
# Cell 2. Environment verification
checks = {'python_executable_raw': PYTHON_EXE, 'openai_api_key_present': bool(os.environ.get('OPENAI_API_KEY')), 'local_model_base_exists': Path(LOCAL_MODEL_BASE).exists(), 'run_ga_search_exists': (SCRIPTS_ROOT / 'run_ga_search.py').exists()}
try:
    import torch
    checks.update({'torch_version': torch.__version__, 'cuda_available': bool(torch.cuda.is_available()), 'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''})
except Exception as exc:
    checks['torch_status'] = f'unavailable: {exc}'
try:
    import transformers
    checks['transformers_version'] = transformers.__version__
except Exception as exc:
    checks['transformers_status'] = f'unavailable: {exc}'
print(json.dumps(checks, indent=2, ensure_ascii=False))

{
  "python_executable_raw": "/home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10",
  "openai_api_key_present": true,
  "local_model_base_exists": true,
  "run_ga_search_exists": true,
  "torch_version": "2.9.1+cu128",
  "cuda_available": true,
  "gpu_name": "NVIDIA RTX A6000",
  "transformers_version": "4.57.3"
}


In [9]:
# Cell 3. Source/flag verification
run_ga = (SCRIPTS_ROOT / 'run_ga_search.py').read_text(encoding='utf-8')
advisor_feedback = (SCRIPTS_ROOT / 'advisor_feedback.py').read_text(encoding='utf-8')
required_flags = ['--cloud-only-advisor-compression','--disable-compression-fallback','--advisor-proposal-k','--advisor-min-usable-compression-proposals','--advisor-repair-invalid-proposals','--advisor-repair-max-attempts','--advisor-require-block-proposal-after-detpass','--advisor-require-token-delta','--advisor-disallow-genome-only-after-detpass','--advisor-cloud-only-strict-schema','--advisor-before-after-report','--advisor-include-dataset-feedback','--advisor-feedback-topk','--advisor-feedback-max-failures-per-family','--advisor-include-prompt-before-after-context','--advisor-include-case-abc-sections','--advisor-write-debug-prompt-sections']
required_sections = ['ADVISOR_ROLE','CURRENT_STATE','DATASET_EVALUATION_FEEDBACK','PROMPT_TOKEN_BREAKDOWN','BLOCK_TOKEN_BREAKDOWN','PROTECTED_BLOCKS','COMPRESSION_ALLOWED_BLOCKS','CASE_A_MICRO_COMPRESSION','CASE_B_THRESHOLD_BLOCK_COMPRESSION','CASE_C_AGGRESSIVE_MULTI_GLOBAL_COMPRESSION','STRICT_JSON_RESPONSE_SCHEMA']
rows = [{'kind':'CLI flag','name':x,'present':x in run_ga} for x in required_flags] + [{'kind':'prompt section','name':x,'present':x in advisor_feedback} for x in required_sections]
rows += [{'kind':'artifact','name':'advisor_cloud_only_summary.csv','present':'advisor_cloud_only_summary.csv' in run_ga}, {'kind':'artifact','name':'advisor_cloud_only_before_after.csv','present':'advisor_cloud_only_before_after.csv' in run_ga}, {'kind':'artifact','name':'advisor_case_ABC_summary.csv','present':'advisor_case_ABC_summary.csv' in run_ga}]
try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    for r in rows: print(r)
missing = [r for r in rows if not r['present']]
assert not missing, missing

,kind,name,present,required_for_by_config
0,required_existing_flag,--llm-mutation-advisor,True,True
1,required_existing_flag,--advisor-model-key,True,True
2,required_existing_flag,--advisor-trigger-mode,True,True
3,required_existing_flag,--advisor-force-child-quota,True,True
4,required_existing_flag,--advisor-min-population-for-child,True,True
...,...,...,...,...
64,optional_strict_prompt_section,COMPRESSION_ALLOWED_BLOCKS,False,False
65,optional_strict_prompt_section,CASE_A_MICRO_COMPRESSION,False,False
66,optional_strict_prompt_section,CASE_B_THRESHOLD_BLOCK_COMPRESSION,False,False
67,optional_strict_prompt_section,CASE_C_AGGRESSIVE_MULTI_GLOBAL_COMPRESSION,False,False


[SOURCE / FLAG VERIFICATION SUMMARY]
STRICT_CLOUD_ONLY_IMPLEMENTED: False

[OK] Existing strong compression flags are available.
[MODE] This notebook can run cloud-only-by-config.
[NOTE] Strict cloud-only flags/sections are optional and currently: not implemented


In [6]:
# Cell 4. Wrapper definition
def run_command(cmd, label):
    print('[' + label + '] ' + ' '.join(map(str, cmd)))
    p = subprocess.run(cmd, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    (ANALYSIS_ROOT / f'{label}_stdout.txt').write_text(p.stdout, encoding='utf-8')
    (ANALYSIS_ROOT / f'{label}_stderr.txt').write_text(p.stderr, encoding='utf-8')
    print(p.stdout[-2000:])
    if p.returncode:
        print(p.stderr[-4000:])
        raise RuntimeError(f'{label} failed: {p.returncode}')
    return p

def run_ga_all_categories(label, *, categories=SMOKE_CATEGORIES, use_advisor=False, cloud_only_advisor_compression=False, disable_compression_fallback=False, advisor_proposal_k=6, advisor_min_usable_compression_proposals=1, advisor_repair_invalid_proposals=False, advisor_repair_max_attempts=0, advisor_require_block_proposal_after_detpass=False, advisor_require_token_delta=False, advisor_disallow_genome_only_after_detpass=False, advisor_cloud_only_strict_schema=False, advisor_before_after_report=True, advisor_include_dataset_feedback=True, advisor_feedback_topk=5, advisor_feedback_max_failures_per_family=3, advisor_include_prompt_before_after_context=True, advisor_include_case_abc_sections=True, advisor_write_debug_prompt_sections=True, mutation_mode='legacy', enable_compression_mutation=False, advisor_compression_child_quota=1, local_compression_quotas_zero=False, run=RUN_SMOKE_COMMANDS):
    out = ANALYSIS_ROOT / label
    cmd = [PYTHON_EXE, '-u', str(SCRIPTS_ROOT / 'run_ga_search.py'), '--profile','version0_15','--model-key',MODEL_KEY,'--population','2','--gens','1','--sample-size','2','--validation-size','2','--cheap-eval-limit','1','--candidate-k','1','--repair-attempts','0','--selection-mode','redesign','--fitness-mode','phase_aware','--stop-controller-mode','active','--compression-detpass-threshold','0','--progress','quiet','--full-run','--force','--output-root',str(out),'--limit-per-category',str(SMOKE_LIMIT_PER_CATEGORY)]
    for c in categories: cmd += ['--category', str(c)]
    if USE_MOCK_LLM: cmd += ['--llm-mode', 'mock']
    if mutation_mode: cmd += ['--mutation-mode', mutation_mode]
    if enable_compression_mutation: cmd += ['--enable-compression-mutation']
    if use_advisor:
        cmd += ['--llm-mutation-advisor','--advisor-model-key',ADVISOR_MODEL_KEY,'--advisor-trigger-mode','always','--advisor-force-child-quota','--advisor-min-population-for-child','2','--advisor-compression-child-quota',str(advisor_compression_child_quota),'--advisor-proposal-k',str(advisor_proposal_k),'--advisor-min-usable-compression-proposals',str(advisor_min_usable_compression_proposals),'--advisor-feedback-topk',str(advisor_feedback_topk),'--advisor-feedback-max-failures-per-family',str(advisor_feedback_max_failures_per_family)]
    flags = {'--cloud-only-advisor-compression':cloud_only_advisor_compression,'--disable-compression-fallback':disable_compression_fallback,'--advisor-repair-invalid-proposals':advisor_repair_invalid_proposals,'--advisor-require-block-proposal-after-detpass':advisor_require_block_proposal_after_detpass,'--advisor-require-token-delta':advisor_require_token_delta,'--advisor-disallow-genome-only-after-detpass':advisor_disallow_genome_only_after_detpass,'--advisor-cloud-only-strict-schema':advisor_cloud_only_strict_schema,'--advisor-before-after-report':advisor_before_after_report,'--advisor-include-prompt-before-after-context':advisor_include_prompt_before_after_context,'--advisor-include-case-abc-sections':advisor_include_case_abc_sections,'--advisor-write-debug-prompt-sections':advisor_write_debug_prompt_sections,'--enable-block-token-breakdown':True,'--allow-aggressive-compression':True,'--aggressive-compression-after-target':True,'--enable-multi-block-compression':True}
    for f, on in flags.items():
        if on: cmd.append(f)
    if advisor_repair_invalid_proposals: cmd += ['--advisor-repair-max-attempts', str(advisor_repair_max_attempts)]
    if local_compression_quotas_zero:
        cmd += ['--compression-child-quota','0','--compression-child-ratio','0.0','--micro-compression-child-quota','0','--micro-compression-child-ratio','0.0','--block-compression-child-quota','0','--block-compression-child-ratio','0.0','--multi-block-compression-child-quota','0','--multi-block-compression-child-ratio','0.0','--global-budget-compression-child-quota','0']
    if run: run_command(cmd, label)
    else: print('[DRY COMMAND]', ' '.join(map(str, cmd)))
    return out, cmd

In [7]:
# Cell 5. Artifact readers
def read_json(path, default=None):
    p = Path(path)
    return json.loads(p.read_text(encoding='utf-8')) if p.exists() else default

def read_jsonl(path):
    p = Path(path)
    return [json.loads(x) for x in p.read_text(encoding='utf-8').splitlines() if x.strip()] if p.exists() else []

def read_csv_rows(path):
    p = Path(path)
    if not p.exists(): return []
    with p.open(newline='', encoding='utf-8') as h: return list(csv.DictReader(h))

def extract_blocks(genome):
    blocks = list((genome or {}).get('blocks') or [])
    return {'core':[b for b in blocks if b in {'01','02'}], 'optional':[b for b in blocks if b not in {'01','02'}]}

def load_progress(run_dir): return read_csv_rows(Path(run_dir) / 'ga_generation_progress.csv')
def load_transitions(run_dir): return read_csv_rows(Path(run_dir) / 'population_transitions.csv')
def load_advisor_proposals(run_dir): return read_jsonl(Path(run_dir) / 'advisor_mutation_proposals.jsonl')
def classify_case_abc(row):
    if row.get('case_label'): return row.get('case_label')
    return {'micro':'Case A','block':'Case B','multi_block':'Case C','global_budget':'Case C'}.get(str(row.get('compression_level') or ''), '')

def inspect_run(run_dir):
    run_dir = Path(run_dir); s = read_json(run_dir / 'ga_summary.json', {}) or {}; m = read_jsonl(run_dir / 'mutation_proposals.jsonl')
    return {'run_dir':str(run_dir),'best_DETPass':s.get('best_DETPass'),'advisor_model_key':s.get('advisor_model_key'),'cloud_only_advisor_compression':s.get('cloud_only_advisor_compression'),'new_by_compression_fallback':s.get('new_by_compression_fallback'),'advisor_unfilled_quota':s.get('advisor_unfilled_quota'),'advisor_rows':len(load_advisor_proposals(run_dir)),'fallback_rows':sum(1 for r in m if r.get('source') == 'compression_fallback'),'transition_rows':len(load_transitions(run_dir))}

def summarize_run(run_dir):
    info = inspect_run(run_dir); print(json.dumps(info, indent=2, ensure_ascii=False)); return info

In [ ]:
# Cell 6. Cloudless baseline smoke
cloudless_dir, cloudless_cmd = run_ga_all_categories('cloudless_baseline_smoke', use_advisor=False, mutation_mode='cloudless_decompiler', enable_compression_mutation=True, advisor_before_after_report=False)
cloudless_summary = summarize_run(cloudless_dir)

In [ ]:
# Cell 7. Hybrid advisor smoke
hybrid_dir, hybrid_cmd = run_ga_all_categories('hybrid_advisor_smoke', use_advisor=True, mutation_mode='hybrid', enable_compression_mutation=True, disable_compression_fallback=False, advisor_before_after_report=True)
hybrid_summary = summarize_run(hybrid_dir)

In [8]:
# Cell 8. Cloud-only advisor smoke
cloud_only_dir, cloud_only_cmd = run_ga_all_categories('cloud_only_advisor_smoke', use_advisor=True, mutation_mode='legacy', enable_compression_mutation=False, cloud_only_advisor_compression=True, disable_compression_fallback=True, advisor_repair_invalid_proposals=True, advisor_repair_max_attempts=2, advisor_require_block_proposal_after_detpass=True, advisor_require_token_delta=True, advisor_disallow_genome_only_after_detpass=True, advisor_cloud_only_strict_schema=True, advisor_before_after_report=True, advisor_compression_child_quota=1, local_compression_quotas_zero=True)
cloud_only_summary = summarize_run(cloud_only_dir)
assert all(r.get('source') != 'compression_fallback' for r in read_jsonl(Path(cloud_only_dir) / 'mutation_proposals.jsonl'))
assert all(str(r.get('fallback_disabled')).lower() == 'true' for r in load_transitions(cloud_only_dir))

[cloud_only_advisor_smoke] /home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10 -u /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile version0_15 --model-key qwen25_coder_14b --population 2 --gens 1 --sample-size 2 --validation-size 2 --cheap-eval-limit 1 --candidate-k 1 --repair-attempts 0 --selection-mode redesign --fitness-mode phase_aware --stop-controller-mode active --compression-detpass-threshold 0 --progress quiet --full-run --force --output-root /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/cloud_only_advisor_analysis_20260612_132600/cloud_only_advisor_smoke --limit-per-category 1 --category 3 --category 4 --llm-mode mock --mutation-mode legacy --llm-mutation-advisor --advisor-model-key gpt41_mini --advisor-trigger-mode always --advisor-force-child-quota --advisor-min-population-for-child 2 --advisor-compression-child-quota 1 --advisor-proposal-k 6 --advisor-min-usable-compression

RuntimeError: cloud_only_advisor_smoke failed: 2

In [ ]:
# Cell 9. Dataset evaluation feedback inspection
feedback_batches = read_jsonl(Path(cloud_only_dir) / 'advisor_feedback_batches.jsonl')
latest_feedback = feedback_batches[-1] if feedback_batches else {}
dataset_feedback = latest_feedback.get('dataset_evaluation_feedback') or {}
print(json.dumps({k: dataset_feedback.get(k) for k in ['evaluated_categories','validation_det_pass_rate','best_so_far_DETPass','avg_prompt_tokens','top_failures','compression_readiness']}, indent=2, ensure_ascii=False))

In [ ]:
# Cell 10. Exact advisor prompt inspection
prompt_files = sorted(Path(cloud_only_dir).glob('advisor_prompt_generation_*.txt'))
prompt_text = prompt_files[-1].read_text(encoding='utf-8') if prompt_files else ''
section_files = sorted(Path(cloud_only_dir).glob('advisor_prompt_case_sections_generation_*.json'))
sections = read_json(section_files[-1], {}) if section_files else {}
print('advisor_model_key:', ADVISOR_MODEL_KEY, 'prompt_chars:', len(prompt_text))
for key in ['CASE_A_MICRO_COMPRESSION','CASE_B_THRESHOLD_BLOCK_COMPRESSION','CASE_C_AGGRESSIVE_MULTI_GLOBAL_COMPRESSION','STRICT_JSON_RESPONSE_SCHEMA','DATASET_EVALUATION_FEEDBACK','BLOCK_TOKEN_BREAKDOWN']:
    excerpt = str(sections.get(key, ''))[:1200]
    (ANALYSIS_ROOT / f'{key.lower()}_excerpt.txt').write_text(excerpt, encoding='utf-8')
    print('
##', key); print(excerpt)

In [ ]:
# Cell 11. Advisor raw response inspection
responses = sorted(Path(cloud_only_dir).glob('advisor_response_generation_*.json'))
latest_response = read_json(responses[-1], {}) if responses else {}
parsed = latest_response.get('parsed') or {}
keys = ['proposals','mutation_proposals','micro_compression_proposals','block_compression_proposals','multi_block_compression_proposals','global_budget_compression_proposals']
print(json.dumps({'advisor_model_key': latest_response.get('advisor_model_key') or parsed.get('advisor_model_key'), 'parsed_keys': sorted(parsed.keys()), 'proposal_array_counts': {k: len(parsed.get(k) or []) for k in keys}, 'accepted_count': len(latest_response.get('accepted_proposals') or []), 'rejected_count': len(latest_response.get('rejected_proposals') or []), 'proposals_repaired': latest_response.get('proposals_repaired')}, indent=2, ensure_ascii=False))

In [ ]:
# Cell 12. Proposal table
proposal_rows = read_csv_rows(Path(cloud_only_dir) / 'advisor_mutation_summary.csv')
for r in proposal_rows: r['case_label'] = classify_case_abc(r)
cols = ['generation','proposal_id','advisor_model_key','schema_source','case_label','operator','selected_block_id','selected_block_ids','expected_token_delta','accepted','rejection_reason','dataset_feedback_reason']
try:
    import pandas as pd
    df = pd.DataFrame(proposal_rows); display(df[[c for c in cols if c in df.columns]])
except Exception:
    for r in proposal_rows: print({c:r.get(c) for c in cols})

In [ ]:
# Cell 13. Before/after block comparison
before_after = read_csv_rows(Path(cloud_only_dir) / 'advisor_cloud_only_before_after.csv')
cols = ['generation','proposal_id','case_label','operator','parent_core_blocks','child_core_blocks','parent_optional_blocks','child_optional_blocks','parent_prompt_tokens','child_prompt_tokens','measured_prompt_token_delta','accepted','applied','rejection_reason']
try:
    import pandas as pd
    df_ba = pd.DataFrame(before_after); display(df_ba[[c for c in cols if c in df_ba.columns]])
except Exception:
    for r in before_after: print({c:r.get(c) for c in cols})
print('accepted block diffs:', len(read_jsonl(Path(cloud_only_dir) / 'advisor_accepted_block_diffs.jsonl')))

In [ ]:
# Cell 14. Before/after prompt token and DETPass comparison
runs = {'cloudless':cloudless_dir, 'hybrid':hybrid_dir, 'cloud_only':cloud_only_dir}
summary_rows = []
for name, rd in runs.items():
    s = read_json(Path(rd) / 'ga_summary.json', {}) or {}; p = load_progress(rd); last = p[-1] if p else {}
    summary_rows.append({'mode':name,'best_DETPass':s.get('best_DETPass'),'last_avg_prompt_tokens':last.get('avg_prompt_tokens') or last.get('tokens'),'advisor_accepted':s.get('advisor_proposals_accepted_applied'),'fallback_rows':s.get('new_by_compression_fallback'),'cloud_only_unfilled_quota':s.get('advisor_unfilled_quota')})
try:
    import pandas as pd, matplotlib.pyplot as plt
    df_summary = pd.DataFrame(summary_rows); display(df_summary)
    ax = df_summary.plot(kind='bar', x='mode', y='best_DETPass', legend=False, title='Best DETPass by mode'); ax.set_ylabel('DETPass'); plt.show()
except Exception:
    for r in summary_rows: print(r)

In [ ]:
# Cell 15. Case A/B/C summary
case_rows = read_csv_rows(Path(cloud_only_dir) / 'advisor_case_ABC_summary.csv')
try:
    import pandas as pd
    display(pd.DataFrame(case_rows))
except Exception:
    for r in case_rows: print(r)

In [ ]:
# Cell 16. Three-way comparison
comparison = []
for mode, rd in runs.items():
    s = read_json(Path(rd) / 'ga_summary.json', {}) or {}; progress = load_progress(rd); tokens = []
    for r in progress:
        try: tokens.append(float(r.get('avg_prompt_tokens') or r.get('tokens') or 0))
        except Exception: pass
    comparison.append({'mode':mode,'best_DETPass':s.get('best_DETPass'),'validation_det_pass_rate':s.get('best_DETPass'),'first_tokens':tokens[0] if tokens else '', 'min_tokens':min(tokens) if tokens else '', 'last_tokens':tokens[-1] if tokens else '', 'token_reduction_ratio':((tokens[0]-min(tokens))/tokens[0]) if tokens and tokens[0] else '', 'advisor_accepted_rows':s.get('advisor_proposals_accepted_applied'), 'fallback_rows':s.get('new_by_compression_fallback'), 'block_diff_rows':len(read_jsonl(Path(rd) / 'ga_block_diffs.jsonl')), 'before_after_rows':len(read_csv_rows(Path(rd) / 'advisor_cloud_only_before_after.csv'))})
try:
    import pandas as pd
    comparison_df = pd.DataFrame(comparison); display(comparison_df)
except Exception:
    for r in comparison: print(r)

In [ ]:
# Cell 17. Final gate
def gate_for_run(mode, rd):
    s = read_json(Path(rd) / 'ga_summary.json', {}) or {}; mut = read_jsonl(Path(rd) / 'mutation_proposals.jsonl'); advisor = load_advisor_proposals(rd)
    g = {'mode':mode, 'detpass_ge_90':float(s.get('best_DETPass') or 0) >= 90, 'avg_prompt_tokens_gt_0':bool(load_progress(rd)), 'fallback_rows':sum(1 for r in mut if r.get('source') == 'compression_fallback'), 'advisor_proposal_rows':len(advisor), 'advisor_accepted_or_unfilled_recorded': bool(s.get('advisor_proposals_accepted_applied') or s.get('advisor_unfilled_quota') is not None), 'before_after_exists':(Path(rd) / 'advisor_cloud_only_before_after.csv').exists(), 'case_summary_exists':(Path(rd) / 'advisor_case_ABC_summary.csv').exists()}
    if mode == 'cloud_only': g['cloud_only_pass'] = g['fallback_rows'] == 0 and g['advisor_proposal_rows'] > 0 and g['advisor_accepted_or_unfilled_recorded'] and g['before_after_exists'] and g['case_summary_exists']
    return g
gates = [gate_for_run('cloudless', cloudless_dir), gate_for_run('hybrid', hybrid_dir), gate_for_run('cloud_only', cloud_only_dir)]
try:
    import pandas as pd
    display(pd.DataFrame(gates))
except Exception:
    for g in gates: print(g)

In [ ]:
# Cell 18. Optional full run command
full_cloud_only_cmd = [PYTHON_EXE,'-u',str(SCRIPTS_ROOT / 'run_ga_search.py'),'--profile','version0_15','--model-key','qwen25_coder_14b','--full-run','--llm-mode','worker','--llm-mutation-advisor','--advisor-model-key','gpt41_mini','--advisor-trigger-mode','always','--cloud-only-advisor-compression','--disable-compression-fallback','--advisor-proposal-k','6','--advisor-min-usable-compression-proposals','2','--advisor-repair-invalid-proposals','--advisor-repair-max-attempts','2','--advisor-require-block-proposal-after-detpass','--advisor-require-token-delta','--advisor-disallow-genome-only-after-detpass','--advisor-cloud-only-strict-schema','--advisor-before-after-report','--advisor-include-dataset-feedback','--advisor-feedback-topk','5','--advisor-feedback-max-failures-per-family','3','--advisor-include-prompt-before-after-context','--advisor-include-case-abc-sections','--advisor-write-debug-prompt-sections','--compression-detpass-threshold','90','--aggressive-compression-after-target','--allow-aggressive-compression','--advisor-compression-child-quota','2','--compression-child-quota','0','--compression-child-ratio','0.0','--micro-compression-child-quota','0','--micro-compression-child-ratio','0.0','--block-compression-child-quota','0','--block-compression-child-ratio','0.0','--multi-block-compression-child-quota','0','--multi-block-compression-child-ratio','0.0','--global-budget-compression-child-quota','0','--enable-block-token-breakdown','--enable-multi-block-compression','--min-compression-token-delta','50']
print('RUN_FULL_CLOUD_ONLY =', RUN_FULL_CLOUD_ONLY)
print(' '.join(map(str, full_cloud_only_cmd)))
if RUN_FULL_CLOUD_ONLY: run_command(full_cloud_only_cmd, 'FULL_cloud_only')

In [ ]:
# Cell 19. Save paper artifacts
paper_dir = ANALYSIS_ROOT / 'paper_artifacts'; paper_dir.mkdir(parents=True, exist_ok=True)
for dst, src in {'cloud_only_main_summary.csv':Path(cloud_only_dir)/'advisor_cloud_only_summary.csv','cloud_only_before_after.csv':Path(cloud_only_dir)/'advisor_cloud_only_before_after.csv','cloud_only_case_ABC_summary.csv':Path(cloud_only_dir)/'advisor_case_ABC_summary.csv'}.items():
    if src.exists(): shutil.copy2(src, paper_dir / dst)
try:
    import pandas as pd
    pd.DataFrame(comparison).to_csv(paper_dir / 'cloudless_vs_hybrid_vs_cloudonly.csv', index=False)
except Exception:
    with (paper_dir / 'cloudless_vs_hybrid_vs_cloudonly.csv').open('w', newline='', encoding='utf-8') as h:
        w = csv.DictWriter(h, fieldnames=sorted({k for r in comparison for k in r})); w.writeheader(); w.writerows(comparison)
(paper_dir / 'advisor_prompt_sections.md').write_text('
'.join([f'## {k}

```text
{str(v)[:4000]}
```
' for k,v in sections.items()]), encoding='utf-8')
(paper_dir / 'advisor_dataset_feedback_examples.md').write_text('# Dataset feedback examples

```json
' + json.dumps(dataset_feedback.get('top_failures', []), ensure_ascii=False, indent=2) + '
```
', encoding='utf-8')
print('paper artifacts:', paper_dir)

In [ ]:
# Cell 20. Interpretation template
lines = [
    '## Korean summary',
    f"- ???? ???: {dataset_feedback.get('top_failure_families')}",
    f'- Advisor model: {ADVISOR_MODEL_KEY}',
    f"- Cloud-only fallback rows: {cloud_only_summary.get('fallback_rows')}",
    f"- Advisor unfilled quota: {cloud_only_summary.get('advisor_unfilled_quota')}",
    f"- Case A/B/C ??: {Path(cloud_only_dir) / 'advisor_case_ABC_summary.csv'}",
    f"- Before/after block ? token ??: {Path(cloud_only_dir) / 'advisor_cloud_only_before_after.csv'}",
    '- ??: mock smoke? ?? token reduction ?? ?? advisor ??? ???? ???. ?? ??? full command? ????? ??? ? artifact?? ????.',
    '',
    '## English summary',
    f"- Dataset feedback sent to the advisor: {dataset_feedback.get('top_failure_families')}",
    f'- Advisor model: {ADVISOR_MODEL_KEY}',
    f"- Cloud-only fallback rows: {cloud_only_summary.get('fallback_rows')}",
    f"- Advisor unfilled quota: {cloud_only_summary.get('advisor_unfilled_quota')}",
    '- Case A/B/C proposal dynamics are in advisor_case_ABC_summary.csv.',
    '- Prompt/genome/block before/after traceability is in advisor_cloud_only_before_after.csv.',
    '- Limitation: mock smoke validates pipeline observability only; it does not prove real token reduction or real cloud advisor contribution.',
]
interpretation = '\n'.join(lines)
print(interpretation)
(ANALYSIS_ROOT / 'interpretation_template.md').write_text(interpretation, encoding='utf-8')